[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/notebooks/EN/chapter5_seminar_notebook.ipynb)

# Time Series Analysis — Seminar 5: Conditional volatility, ARCH and GARCH

*Seminar notebook. Daniel Traian Pele, Bucharest University of Economic Studies.*

- The seminar comes before Lecture 5; the slides give the definitions ("What you need today").
- [Solved] tasks: full solution here and in the slides; [Proposed] tasks: write your own code in the empty cell.
- The first code cell installs the `arch` package if it is missing (`pip install arch`).
- The seminar is for practice and is not graded; the solutions of [Proposed] tasks are discussed in class.

> The solutions are discussed in the seminar.

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/TSA/Chapter_5'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# The arch package (ARCH/GARCH estimation) is not preinstalled in Google Colab: pip install arch
import importlib.util, subprocess, sys
if importlib.util.find_spec('arch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'arch'])

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

- Transparent background, legend below the plot, course palette.

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import tsa_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

- Daily market data from EODHD, saved in `data/market` of the TSA repository; EUR/RON is the official BNR reference rate.
- Macroeconomic series: FRED (St. Louis Fed), Eurostat and the classic data sets of statsmodels.
- Returns in %: $r_t = 100(\ln P_t - \ln P_{t-1})$; each series on its own calendar.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the TSA repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Seminar functions

- Helpers of Chapter 5 (`returns`, `arch_lm`, `ljung_box`, `fit`, `par`, `sig`, `summary`, `ols_ar1`, `oos_forecasts`, `qlike`, `dm_test`) and the seminar functions `garch_algebra`, `archlm_by_hand`, `b1_estimate`, `b3_asymmetry`, `b5_oos`.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox
from arch import arch_model
SEED = 2026
ASSETS = ['sp500', 'bet', 'eurron', 'btc']
NAME = {'sp500': 'S&P 500', 'bet': 'BET', 'eurron': 'EUR/RON', 'btc': 'Bitcoin'}
START = {'sp500': '2000-01-01', 'bet': '2000-01-01', 'eurron': None, 'btc': None}
COLORS = {'sp500': '#1A3A6E', 'bet': '#CD0000', 'eurron': '#2E7D32', 'btc': '#B5853F'}
SCALE = {'eurron': 10}
EPISODES = {'2008': ('2008-09-01', '2009-03-31'), '2020': ('2020-02-15', '2020-05-31')}
OOS_START = '2015-01-01'
REFIT = 250
LAMBDA = 0.94
ALPHA_VAR = 0.01
LB_LAGS = 10
ARCH_LAGS = 5
IG = 0.9995
TERM_DATES = ['2017-11-03', '2020-03-16']


def returns(k, start=None, end=None):
    """Daily log returns in % on the series' own calendar."""
    r = log_returns(k, start or START.get(k)) if end is None else log_returns(k, start or START.get(k), end)
    return r.rename(k)


def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def acf_vals(x, nlags):
    """Sample autocorrelations rho_hat(1..nlags) = sum (x_t - m)(x_{t-k} - m) / sum (x_t - m)^2."""
    e = np.asarray(x, float) - np.mean(x)
    s = np.sum(e ** 2)
    return np.array([np.sum(e[k:] * e[:-k]) / s for k in range(1, nlags + 1)])


def ljung_box(x, lags=LB_LAGS, dof=0):
    """Ljung-Box Q(lags) and its p-value (chi-square with lags - dof degrees of freedom)."""
    lb = acorr_ljungbox(pd.Series(np.asarray(x, float)).dropna(), lags=[lags], model_df=dof)
    return float(lb['lb_stat'].iloc[0]), float(lb['lb_pvalue'].iloc[0])


def arch_lm(x, q=ARCH_LAGS):
    """Engle's (1982) ARCH-LM test: regress e_t^2 on a constant and e_{t-1}^2, ..., e_{t-q}^2 (e_t = x_t - mean);
    LM = n R^2 ~ chi-square(q) under no ARCH effects (n = number of regression observations)."""
    e2 = (np.asarray(x, float) - np.mean(x)) ** 2
    y = e2[q:]
    X = np.column_stack([np.ones(len(y))] + [e2[q - j:len(e2) - j] for j in range(1, q + 1)])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    u = y - X @ b
    r2 = 1 - np.sum(u ** 2) / np.sum((y - y.mean()) ** 2)
    lm = len(y) * r2
    return {'lm': float(lm), 'r2': float(r2), 'n': len(y), 'p': float(stats.chi2.sf(lm, q)),
            'crit': float(stats.chi2.ppf(0.95, q)), 'b': [float(v) for v in b]}


def fit(r, vol='GARCH', dist='t', o=0, p=1, q=1, mean='Constant', lags=1, last_obs=None, cov_type='robust'):
    """ML estimation with the arch package: constant or AR(lags) mean; GARCH (o = 0), GJR-GARCH (o = 1), EGARCH,
    ARCH(p). The series is multiplied by SCALE (EUR/RON: 10) and the result remembers the factor in res.sc."""
    sc = SCALE.get(r.name, 1)
    kw = dict(mean=mean, lags=lags if mean == 'AR' else 0, dist=dist, rescale=False)
    if vol == 'ARCH':
        am = arch_model(sc * r, vol='ARCH', p=p, **kw)
    elif vol == 'EGARCH':
        am = arch_model(sc * r, vol='EGARCH', p=1, o=1, q=1, **kw)
    else:
        am = arch_model(sc * r, vol='GARCH', p=p, o=o, q=q, **kw)
    res = am.fit(disp='off', last_obs=last_obs, cov_type=cov_type, options={'maxiter': 2000})
    res.sc = sc
    return res


def par(res):
    """Parameters in % units (the estimation scale removed): mu and Const / sc, omega / sc^2 (GARCH, GJR, ARCH)."""
    p = res.params.copy()
    for n in p.index:
        if n in ('mu', 'Const'):
            p[n] = p[n] / res.sc
        elif n == 'omega' and res.model.volatility.__class__.__name__ != 'EGARCH':
            p[n] = p[n] / res.sc ** 2
    return p


def sig(res):
    """Conditional volatility in %."""
    return res.conditional_volatility / res.sc


def persistence(res):
    """alpha + beta (GARCH), alpha + beta + gamma/2 (GJR with symmetric innovations), beta (EGARCH)."""
    p = res.params
    if res.model.volatility.__class__.__name__ == 'EGARCH':
        return float(p['beta[1]'])
    return float(sum(v for n, v in p.items() if n.startswith(('alpha', 'beta'))) + 0.5 * p.get('gamma[1]', 0.0))


def half_life(pers):
    """Days after which half of a variance shock has disappeared: ln 0.5 / ln(persistence)."""
    return float(np.log(0.5) / np.log(pers)) if 0 < pers < 1 else float('inf')


def summary(res, r):
    """Parameters and robust standard errors in % units, persistence, half-life, long-run and sample volatility."""
    p = par(res)
    se = res.std_err.copy()
    for n in se.index:
        if n in ('mu', 'Const'):
            se[n] = se[n] / res.sc
        elif n == 'omega':
            se[n] = se[n] / res.sc ** 2
    pers = persistence(res)
    ppy = periods_per_year(r)
    adj = res.nobs * np.log(res.sc)               # log-likelihood of r_t = log-likelihood of sc r_t + n ln sc
    out = {'n': int(res.nobs), 'first': r.index[0].date().isoformat(), 'ppy': float(ppy),
           'params': {k: float(v) for k, v in p.items()}, 'se': {k: float(v) for k, v in se.items()},
           'loglik': float(res.loglikelihood + adj), 'aic': float(res.aic - 2 * adj), 'bic': float(res.bic - 2 * adj),
           'k': int(len(p)), 'pers': pers, 'hl': half_life(pers), 'vol_sample': float(r.std() * np.sqrt(ppy))}
    if 'omega' in p and res.model.volatility.__class__.__name__ != 'EGARCH' and pers < IG:
        uv = p['omega'] / (1 - pers)
        out['uv'] = float(uv)
        out['vol_lr'] = float(np.sqrt(uv * ppy))
    return out


def ols_ar1(r):
    """AR(1) for the mean by OLS: r_t = c + phi r_{t-1} + e_t; coefficients, classic and HAC-robust t statistics."""
    y, x = r.values[1:], r.values[:-1]
    X = sm.add_constant(x)
    o = sm.OLS(y, X).fit()
    h = sm.OLS(y, X).fit(cov_type='HC0')
    return {'c': float(o.params[0]), 'phi': float(o.params[1]), 'se': float(o.bse[1]), 'se_robust': float(h.bse[1]),
            'resid': pd.Series(o.resid, index=r.index[1:])}


def ar_order(r, pmax=5):
    """AR(p) for the mean with constant variance, p = 0..pmax chosen by BIC on the same observations."""
    bic = {}
    for p in range(pmax + 1):
        am = arch_model(r * SCALE.get(r.name, 1), mean='AR' if p else 'Constant', lags=p, vol='Constant',
                        hold_back=pmax, rescale=False)
        bic[p] = float(am.fit(disp='off').bic)
    return min(bic, key=bic.get), bic


def ewma_variance(r, lam=LAMBDA, init=250):
    """EWMA variance forecast for day t made at t-1: h_t = lam h_{t-1} + (1 - lam) r_{t-1}^2 (zero mean)."""
    x = np.asarray(r, float)
    h = np.empty(len(x))
    h[0] = np.mean(x[:init] ** 2)
    for t in range(1, len(x)):
        h[t] = lam * h[t - 1] + (1 - lam) * x[t - 1] ** 2
    return pd.Series(h, index=r.index)


def news_impact(res, eps, s2bar):
    """sigma_t^2 as a function of the shock eps_{t-1}, with sigma_{t-1}^2 fixed at s2bar (Engle and Ng, 1993);
    for series estimated without scaling (SCALE = 1)."""
    p = res.params
    if res.model.volatility.__class__.__name__ == 'EGARCH':
        z = eps / np.sqrt(s2bar)
        return np.exp(p['omega'] + p['alpha[1]'] * (np.abs(z) - np.sqrt(2 / np.pi)) + p['gamma[1]'] * z
                      + p['beta[1]'] * np.log(s2bar))
    return p['omega'] + (p['alpha[1]'] + p.get('gamma[1]', 0.0) * (eps < 0)) * eps ** 2 + p['beta[1]'] * s2bar


def sign_bias_test(z, eps):
    """Engle-Ng (1993) sign-bias tests: z_t^2 on a constant, S-_{t-1}, S-_{t-1} eps_{t-1}, S+_{t-1} eps_{t-1};
    t statistics and the joint test T R^2 ~ chi2(3)."""
    d = pd.concat([pd.Series(np.asarray(z)), pd.Series(np.asarray(eps))], axis=1, keys=['z', 'e']).dropna()
    zz, e = d['z'].values, d['e'].values
    y = zz[1:] ** 2
    el = e[:-1]
    sneg = (el < 0).astype(float)
    X = np.column_stack([np.ones_like(el), sneg, sneg * el, (1 - sneg) * el])
    ols = sm.OLS(y, X).fit()
    joint = len(y) * ols.rsquared
    return {'sign_t': float(ols.tvalues[1]), 'neg_t': float(ols.tvalues[2]), 'pos_t': float(ols.tvalues[3]),
            'joint': float(joint), 'joint_p': float(stats.chi2.sf(joint, 3))}


def oos_forecasts(k, oos_start=OOS_START, refit=REFIT):
    """One-day-ahead variance forecasts on the out-of-sample period: GARCH-t and GJR-t re-estimated every `refit`
    observations on an expanding window (the forecast for day t uses data up to t-1), and EWMA; the one-day VaR 1% of
    GARCH-t and of EWMA-Normal (all in %)."""
    r = returns(k)
    sc = SCALE.get(k, 1)
    i0 = r.index.searchsorted(pd.Timestamp(oos_start))
    cols = {'garch': [], 'gjr': [], 'var_garch': []}
    for s in range(i0, len(r), refit):
        e = min(s + refit, len(r))
        fits = {}
        for lab, o in [('garch', 0), ('gjr', 1)]:
            am = arch_model(sc * r.iloc[:e], mean='Constant', vol='GARCH', p=1, o=o, q=1, dist='t', rescale=False)
            res = am.fit(disp='off', last_obs=r.index[s], options={'maxiter': 2000})
            fixed = am.fix(res.params)
            cols[lab].append(fixed.conditional_volatility.iloc[s:e] ** 2 / sc ** 2)
            fits[lab] = res.params
        p = fits['garch']
        q = stats.t.ppf(ALPHA_VAR, p['nu']) * np.sqrt((p['nu'] - 2) / p['nu'])
        cols['var_garch'].append(-(p['mu'] / sc + np.sqrt(cols['garch'][-1]) * q))
    df = pd.DataFrame({c: pd.concat(v) for c, v in cols.items()})
    df['ewma'] = ewma_variance(r).loc[df.index]
    df['var_ewma'] = -np.sqrt(df['ewma']) * stats.norm.ppf(ALPHA_VAR)
    df['r'] = r.loc[df.index]
    return df


def qlike(proxy, h):
    """QLIKE loss (Patton, 2011), written as proxy / h + ln h: it differs from proxy / h - ln(proxy / h) - 1 only by a
    term that does not depend on the forecast, and it allows a zero proxy."""
    return proxy / h + np.log(h)


def dm_test(la, lb, lags=5):
    """Diebold-Mariano: d = L_a - L_b; HAC (Newey-West) t statistic of the mean of d (negative: model a better)."""
    d = np.asarray(la) - np.asarray(lb)
    ols = sm.OLS(d, np.ones_like(d)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    t = float(ols.tvalues[0])
    return {'mean': float(d.mean()), 't': t, 'p': float(2 * stats.norm.sf(abs(t)))}


def garch_algebra(omega, alpha, beta, s2_t, r_t, mu=0.0, ppy=252):
    """GARCH(1,1) with daily returns in %: persistence, unconditional variance, annualised long-run volatility,
    half-life, and the one-step update sigma^2_{t+1} = omega + alpha (r_t - mu)^2 + beta sigma^2_t."""
    pers = alpha + beta
    uv = omega / (1 - pers)
    return {'pers': pers, 'uv': uv, 'vol_lr': float(np.sqrt(uv * ppy)), 'hl': half_life(pers),
            's2_next': omega + alpha * (r_t - mu) ** 2 + beta * s2_t}


def archlm_by_hand(r2, n, q=5, Q=None, m=10, omega=0.5, alpha=0.5):
    """ARCH-LM from regression output: LM = n R^2 against the chi-square(q) critical value; a Ljung-Box statistic Q
    of the squared residuals against chi-square(m); unconditional variance and kurtosis of an ARCH(1)."""
    lm = n * r2
    out = {'lm': lm, 'crit': float(stats.chi2.ppf(0.95, q)), 'p': float(stats.chi2.sf(lm, q)),
           'uv': omega / (1 - alpha), 'kurt': 3 * (1 - alpha ** 2) / (1 - 3 * alpha ** 2) if 3 * alpha ** 2 < 1 else float('inf')}
    if Q is not None:
        out.update(Q=Q, Qcrit=float(stats.chi2.ppf(0.95, m)), Qp=float(stats.chi2.sf(Q, m)))
    return out


def b1_estimate(k='bet', save=True):
    """ARCH effects and an AR(1)-GARCH(1,1)-t model: ARCH-LM(5) and Ljung-Box Q(10) of the squared residuals of an
    AR(1) mean model; AR(1)-GARCH(1,1)-t estimates with robust standard errors; persistence, half-life, long-run and
    sample volatility; Q(10) of z and z^2; volatility peaks in 2008 and 2020; chart of the returns since 2018 with
    +/- 2 conditional standard deviations."""
    r = returns(k)
    o = ols_ar1(r)
    e = o['resid'].values
    res = fit(r, mean='AR', dist='t')
    out = summary(res, r)
    out['ar_name'] = [n for n in res.params.index if n.endswith('[1]') and n not in ('alpha[1]', 'beta[1]')][0]
    out['lm_e'] = arch_lm(e)
    out['lb_e2'] = ljung_box(e ** 2)
    z = res.std_resid.dropna()
    out['lb_z'] = ljung_box(z, dof=1)
    out['lb_z2'] = ljung_box(z ** 2)
    out['phi_ols'] = o['phi']
    out['se_ols'] = o['se']
    ann = np.sqrt(periods_per_year(r))
    v = sig(res)
    for ep, (a, b) in {'2008': ('2008-09-01', '2009-03-31'), '2020': ('2020-02-15', '2020-05-31')}.items():
        w = v.loc[a:b] * ann
        if len(w) and pd.Timestamp(a) > r.index[0]:
            out[f'peak{ep}'] = float(w.max())
            out[f'date{ep}'] = w.idxmax().date().isoformat()
    out['last_vol'] = float(v.iloc[-1] * ann)
    p = par(res)
    m = p['Const'] + p[out['ar_name']] * r.shift(1)
    out['share_out'] = float((np.abs(r - m) > 2 * v).iloc[1:].mean())
    if save:
        a = '2018-01-01'
        fig, ax = plt.subplots(figsize=(10, 3.8))
        ax.plot(r.loc[a:].index, r.loc[a:], color=COLORS[k], lw=0.6, label=f'{NAME[k]}: daily log return (%)')
        ax.plot(v.loc[a:].index, 2 * v.loc[a:], color=st.MainBlue, lw=1.1, label='+/- 2 conditional standard deviations, AR(1)-GARCH(1,1)-t')
        ax.plot(v.loc[a:].index, -2 * v.loc[a:], color=st.MainBlue, lw=1.1)
        ax.set_ylabel('%')
        st.legend_outside_bottom(ax, ncol=2, y=-0.16)
        st.check_no_grey(fig)
        st.save_fig('ch5_sem_b1')
    return out


def b3_asymmetry(k='sp500', save=True):
    """GARCH-t against GJR-t: gamma with its robust t statistic, the likelihood-ratio test, BIC, the sign-bias test on
    the GARCH-t residuals, and the GJR variance after shocks of -2% and +2%; chart of the two news impact curves."""
    r = returns(k)
    rg, rj = fit(r, dist='t'), fit(r, o=1, dist='t')
    lr = 2 * (rj.loglikelihood - rg.loglikelihood)
    s2bar = float(r.var())
    sc = rj.sc
    nic = {e: float(news_impact(rj, np.array([e * sc]), s2bar * sc ** 2)[0] / sc ** 2) for e in (-2.0, 2.0)}
    out = {'gamma': float(rj.params['gamma[1]']), 't_gamma': float(rj.tvalues['gamma[1]']),
           'alpha_gjr': float(rj.params['alpha[1]']), 'beta_gjr': float(rj.params['beta[1]']),
           'pers_gjr': persistence(rj), 'pers_garch': persistence(rg), 'lr': float(lr), 'p_lr': float(stats.chi2.sf(lr, 1)),
           'bic_garch': float(rg.bic), 'bic_gjr': float(rj.bic), 'sb': sign_bias_test(rg.std_resid, rg.resid),
           'nic_neg': nic[-2.0], 'nic_pos': nic[2.0], 'ratio': nic[-2.0] / nic[2.0], 's2bar': s2bar}
    if save:
        g = np.linspace(-4, 4, 201)
        fig, ax = plt.subplots(figsize=(10, 3.8))
        ax.plot(g, news_impact(rg, g, s2bar), color=st.MainBlue, lw=1.6, label='GARCH(1,1)-t')
        ax.plot(g, news_impact(rj, g, s2bar), color=st.IDAred, lw=1.6, label='GJR-GARCH(1,1)-t')
        ax.set_xlabel('shock yesterday, eps(t-1) (%)')
        ax.set_ylabel('variance today (%^2)')
        st.legend_outside_bottom(ax, ncol=2, y=-0.2)
        st.check_no_grey(fig)
        st.save_fig('ch5_sem_b3')
    return out


def b5_oos(k='sp500', save=True):
    """Out-of-sample one-day forecasts since 2015 (GARCH-t re-estimated every 250 days, EWMA lambda = 0.94): mean
    QLIKE, the Diebold-Mariano test, the VaR 1% exceedances; chart of the cumulative QLIKE difference."""
    df = oos_forecasts(k)
    px = df['r'] ** 2
    lg, le = qlike(px, df['garch']), qlike(px, df['ewma'])
    eg, ee = df['r'] < -df['var_garch'], df['r'] < -df['var_ewma']
    d = le - lg
    out = {'n': int(len(df)), 'first': df.index[0].date().isoformat(), 'q_garch': float(lg.mean()), 'q_ewma': float(le.mean()),
           'dm': dm_test(lg, le), 'nexc_garch': int(eg.sum()), 'nexc_ewma': int(ee.sum()),
           'exc_garch': float(eg.mean()), 'exc_ewma': float(ee.mean()), 'expected': float(0.01 * len(df)),
           'last_var_garch': float(df['var_garch'].iloc[-1]), 'last_var_ewma': float(df['var_ewma'].iloc[-1]),
           'max_day': d.idxmax().date().isoformat(), 'share_max': float(d.max() / d.sum()),
           'q_garch_wo': float(lg.drop(d.idxmax()).mean()), 'q_ewma_wo': float(le.drop(d.idxmax()).mean())}
    if save:
        c = d.cumsum()
        fig, ax = plt.subplots(figsize=(10, 3.6))
        ax.plot(c.index, c.values, color=COLORS[k], lw=1.4, label=f'{NAME[k]}: cumulative QLIKE(EWMA) - QLIKE(GARCH-t)')
        ax.axhline(0, color='black', lw=0.8, ls='--')
        ax.set_ylabel('cumulative loss difference')
        st.legend_outside_bottom(ax, ncol=1, y=-0.16)
        st.check_no_grey(fig)
        st.save_fig('ch5_sem_b5')
    return out

# Part A: computations on paper

- Do each computation on paper first; then check it with the code.

## A1 [Solved]: GARCH(1,1) algebra

**Context.** Daily returns in %: $\omega = 0.02$, $\alpha = 0.08$, $\beta = 0.90$, $\mu = 0$, 252 trading days per year.

1. Compute the persistence and the long-run variance.
2. Compute the annualised long-run volatility.
3. Compute the half-life of a variance shock.
4. Today $\sigma_t^2 = 1.5$ and $r_t = -3\%$; compute $\sigma_{t+1}^2$ and $\sigma_{t+1}$.

**Report:** four numbers and one sentence.

In [8]:
# Solution
garch_algebra(0.02, 0.08, 0.90, s2_t=1.5, r_t=-3.0)

{'pers': 0.98,
 'uv': 0.9999999999999991,
 'vol_lr': 15.874507866387537,
 'hl': 34.309618491520645,
 's2_next': 2.09}

**Interpretation of the result.** Persistence 0.98, long-run variance 1 (15.9% per year), half-life about 34 days; a fall of 3% lifts tomorrow's variance from 1.5 to 2.09.

## A2 [Proposed]: a more reactive market

**Context.** $\omega = 0.05$, $\alpha = 0.12$, $\beta = 0.85$, $\mu = 0$; today $\sigma_t^2 = 2.0$ and $r_t = +1.5\%$. Model: A1.

1. Compute the persistence, the long-run variance and the annualised long-run volatility.
2. Compute the half-life.
3. Compute $\sigma_{t+1}^2$.
4. Write EWMA with $\lambda = 0.94$ as a GARCH(1,1) and say which of the quantities in 1–2 exist for it.

**Report:** four numbers and two sentences.

In [ ]:
# Your code here


## A3 [Solved]: forecasts and VaR 1%

**Context.** The model of A1, with $\sigma_{t+1}^2 = 2.09$ and Normal innovations.

1. Compute $E_t[\sigma_{t+5}^2]$ and $E_t[\sigma_{t+10}^2]$.
2. Compute the 10-day variance and volatility.
3. Compare with the square-root-of-time rule $\sqrt{10}\,\sigma_{t+1}$ and with $\sqrt{10\,\bar\sigma^2}$.
4. Compute the one-day and the 10-day VaR 1%.

**Report:** six numbers and one sentence.

In [10]:
# Solution
s2n = garch_algebra(0.02, 0.08, 0.90, 1.5, -3.0)['s2_next']
pers, s2bar = 0.98, 1.0
f = {h: s2bar + pers ** (h - 1) * (s2n - s2bar) for h in (5, 10)}
sum10 = sum(s2bar + pers ** (h - 1) * (s2n - s2bar) for h in range(1, 11))
q = stats.norm.ppf(0.01)
print('forecasts:', {h: round(v, 3) for h, v in f.items()})
print('10-day variance:', round(sum10, 2), ' volatility:', round(np.sqrt(sum10), 2))
print('sqrt-of-time:', round(np.sqrt(10 * s2n), 2), ' long run:', round(np.sqrt(10 * s2bar), 2))
print('VaR 1%: one day', round(-q * np.sqrt(s2n), 2), ' 10 days', round(-q * np.sqrt(sum10), 2))

forecasts: {5: 2.005, 10: 1.909}
10-day variance: 19.97  volatility: 4.47
sqrt-of-time: 4.57  long run: 3.16
VaR 1%: one day 3.36  10 days 10.4


**Interpretation of the result.** The high variance decays slowly towards 1: the 10-day volatility lies between the square-root-of-time value and the long-run value, closer to the first.

## A4 [Proposed]: forecasts with Student-t innovations

**Context.** The model of A2, with $\sigma_{t+1}^2 = 2.02$ and standardised Student-t innovations with $\nu = 5$. Model: A3.

1. Compute $E_t[\sigma_{t+10}^2]$ and the 10-day variance.
2. Compute the quantile $q_{0.01}$ of the standardised t.
3. Compute the one-day and the 10-day VaR 1% with t and with Normal innovations.

**Report:** six numbers and one sentence.

In [ ]:
# Your code here


## A5 [Solved]: an ARCH-LM test from regression output

**Context.** An AR(1) is fitted to the daily returns of a stock index. The regression of $\hat\varepsilon_t^2$ on a constant and five lags uses $n = 1000$ observations and gives $R^2 = 0.062$; the Ljung–Box statistic of $\hat\varepsilon_t^2$ is $Q(10) = 95.3$.

1. Compute the ARCH-LM statistic and compare it with the 5% value of $\chi^2(5)$.
2. Compare $Q(10)$ with the 5% value of $\chi^2(10)$.
3. Say which model you would fit next.
4. For an ARCH(1) with $\omega = \alpha = 0.5$ and Normal $z_t$, compute the unconditional variance and the kurtosis.

**Report:** two decisions, two numbers and one sentence.

In [12]:
# Solution
archlm_by_hand(r2=0.062, n=1000, q=5, Q=95.3, m=10, omega=0.5, alpha=0.5)

{'lm': 62.0,
 'crit': 11.070497693516351,
 'p': 4.689399882560058e-12,
 'uv': 1.0,
 'kurt': 9.0,
 'Q': 95.3,
 'Qcrit': 18.307038053275146,
 'Qp': 4.732803012479014e-16}

**Interpretation of the result.** LM = 62 and Q(10) = 95.3 are far above 11.07 and 18.31: strong ARCH effects, so the next model is an AR(1)-GARCH(1,1). The ARCH(1) of part 4 has variance 1 and kurtosis 9: Normal innovations, heavy-tailed returns.

## A6 [Proposed]: good and bad news

**Context.** GJR-GARCH: $\omega = 0.02$, $\alpha = 0.03$, $\gamma = 0.12$, $\beta = 0.90$; EGARCH: $\omega = 0$, $\alpha = 0.12$, $\gamma = -0.08$, $\beta = 0.98$; in both $\sigma_t^2 = 1$, $\mu = 0$. Model: A1.

1. Compute the GJR $\sigma_{t+1}^2$ after $\varepsilon_t = -2$ and after $\varepsilon_t = +2$, and their ratio.
2. Compute the GJR persistence and half-life.
3. Compute the EGARCH $\sigma_{t+1}^2$ for the same two shocks.

**Report:** six numbers and one sentence.

In [ ]:
# Your code here


# Part B: real data and interpretation

## B1 [Solved]: ARCH effects and AR(1)-GARCH(1,1)-t for the BET

**Question.** Does the BET have ARCH effects, how persistent is its volatility, and where is it today relative to its long-run level?

1. Fit an AR(1) for the mean by OLS, then run ARCH-LM(5) and Ljung–Box $Q(10)$ on its squared residuals.
2. Estimate AR(1)-GARCH(1,1)-t and report the parameters with robust standard errors.
3. Compute the persistence, the half-life and the annualised long-run volatility, and compare them with the sample volatility.
4. Check $Q(10)$ of $\hat z_t$ and of $\hat z_t^2$, and draw the returns since 2018 with $\pm 2\hat\sigma_t$ bands.
5. Interpretation: is the BET volatility on the last day above or below its long-run level, and what does the model expect for the next months?

**Report:** two test statistics, a table of six parameters, five numbers, the chart and two sentences.

In [14]:
# Solution
b1 = b1_estimate('bet', save=False)
print('ARCH-LM(5) on the AR(1) residuals:', round(b1['lm_e']['lm'], 1), ' Q(10) of squares:', round(b1['lb_e2'][0], 1))
display(pd.DataFrame({'estimate': b1['params'], 'robust SE': b1['se']}).round(4))
print('persistence', round(b1['pers'], 4), ' half-life', round(b1['hl'], 1), ' long-run vol', round(b1['vol_lr'], 1), ' sample vol', round(b1['vol_sample'], 1), ' last day', round(b1['last_vol'], 1))
print('Q(10) z:', b1['lb_z'], ' Q(10) z^2:', b1['lb_z2'])

ARCH-LM(5) on the AR(1) residuals: 1017.1  Q(10) of squares: 2495.7


,estimate,robust SE
Const,0.0766,0.0098
bet[1],0.0907,0.0133
omega,0.0441,0.0085
alpha[1],0.1885,0.0234
beta[1],0.8012,0.0227
nu,5.0859,0.3203


persistence 0.9898  half-life 67.4  long-run vol 32.8  sample vol 22.1  last day 26.2
Q(10) z: (28.62754912492281, 0.0007487129422568828)  Q(10) z^2: (7.217405745140573, 0.7047734675910778)


**Interpretation of the result.** The BET volatility on the last day is below its long-run level, so the model expects it to rise slowly; the long-run level is imprecise because $1 - \alpha - \beta$ is about 0.01.

## B2 [Proposed]: EUR/RON and Bitcoin

**Question.** Do the EUR/RON rate and Bitcoin behave like the BET? Model: B1.

1. Repeat steps 1–4 of B1 for both series (`b1_estimate('eurron', save=False)` estimates EUR/RON on $10\,r_t$).
2. Report $\hat\alpha + \hat\beta$ and say whether the half-life and the long-run volatility exist.
3. Interpretation: what does $\hat\alpha + \hat\beta = 1$ mean for a forecast of the volatility one year ahead?

**Report:** one table and two sentences.

In [ ]:
# Your code here


## B3 [Solved]: the leverage effect in the S&P 500

**Question.** Do falls raise the volatility of the S&P 500 more than rises of the same size?

1. Estimate GARCH(1,1)-t and GJR-GARCH(1,1)-t; report $\gamma$ with its robust $t$ statistic.
2. Compute the LR statistic of GJR against GARCH, its p-value and the two BIC values.
3. Run the sign-bias test on the standardised residuals of GARCH-t.
4. Draw both news impact curves and compute the GJR variance after shocks of −2% and +2%.
5. Interpretation: how does the leverage effect change the VaR on the day after a fall?

**Report:** four statistics with p-values, two variances, the chart and two sentences.

In [16]:
# Solution
b3 = b3_asymmetry('sp500', save=False)
b3

{'gamma': 0.20518725729389392,
 't_gamma': 10.027735545530913,
 'alpha_gjr': 0.0,
 'beta_gjr': 0.8808581501089234,
 'pers_gjr': 0.9834517787558704,
 'pers_garch': 0.9947120452816396,
 'lr': 235.65251563321726,
 'p_lr': 3.4888848674232973e-53,
 'bic_garch': 18169.006314276925,
 'bic_gjr': 17942.165748821248,
 'sb': {'sign_t': 3.5644971855836114,
  'neg_t': 1.1291832329231297,
  'pos_t': -2.612821504737135,
  'joint': 40.66475977251353,
  'joint_p': 7.70229184039106e-09},
 'nic_neg': 2.135562479279996,
 'nic_pos': 1.3148134501044202,
 'ratio': 1.6242323039138316,
 's2bar': 1.4714407863451462}

**Interpretation of the result.** Only falls raise the variance ($\hat\alpha = 0$); after a fall of 2% the variance is about 1.6 times the variance after a rise of 2%, so the VaR is about 1.27 times larger.

## B4 [Proposed]: asymmetry in the BET, EUR/RON and Bitcoin

**Question.** Is there a leverage effect in the BET, in EUR/RON and in Bitcoin? Model: B3.

1. Estimate GJR-GARCH(1,1)-t and EGARCH(1,1)-t; report both values of $\gamma$ with robust $t$ statistics.
2. Compute the LR test of GJR against GARCH and the change in BIC.
3. Interpretation: for EUR/RON a positive return means a weaker leu; what is "bad news" for this series?

**Report:** one table and two sentences.

In [ ]:
# Your code here


## B5 [Solved]: GARCH against EWMA for the S&P 500

**Question.** Does GARCH(1,1)-t forecast tomorrow's variance of the S&P 500 better than EWMA, and is its VaR 1% exceeded on 1% of the days?

1. Re-estimate GARCH(1,1)-t every 250 days on all data up to that day and compute the one-day forecasts $h_t$; compute EWMA with $\lambda = 0.94$.
2. Compute the mean QLIKE of both models and the Diebold–Mariano $t$ statistic.
3. Count the exceedances of the GARCH-t VaR 1% and of the EWMA-Normal VaR 1%, and compare them with the expected number.
4. Draw the cumulative difference of the losses.
5. Interpretation: which model would you give to a risk manager, and what would you still improve?

**Report:** four numbers, two counts, the chart and two sentences.

In [18]:
# Solution
b5 = b5_oos('sp500', save=False)
b5

{'n': 2944,
 'first': '2015-01-02',
 'q_garch': 0.7611897391103296,
 'q_ewma': 0.818761373078798,
 'dm': {'mean': -0.057571633968468254,
  't': -3.6888682742772287,
  'p': 0.00022525379812602132},
 'nexc_garch': 49,
 'nexc_ewma': 65,
 'exc_garch': 0.016644021739130436,
 'exc_ewma': 0.022078804347826088,
 'expected': 29.44,
 'last_var_garch': 1.8449625790667268,
 'last_var_ewma': 1.5681890998790926,
 'max_day': '2018-10-10',
 'share_max': 0.14377321929153308,
 'q_garch_wo': 0.7462853151366233,
 'q_ewma_wo': 0.7955964396521921}

**Interpretation of the result.** GARCH-t has the smaller mean QLIKE and the DM test rejects equal accuracy; both VaR models are exceeded more often than 1%, GARCH-t less so: add the leverage effect (B3).

## B6 [Proposed]: GARCH against EWMA for the BET and EUR/RON

**Question.** Does the result of B5 hold for the BET and for EUR/RON? Model: B5.

1. Compute the out-of-sample GARCH(1,1)-t and EWMA forecasts as in B5.
2. Compute the mean QLIKE of both models and the DM test.
3. Find the day with the largest loss difference, and recompute the mean QLIKE without it.
4. Count the VaR 1% exceedances of both models.
5. Interpretation: why can the DM test be insignificant when one mean QLIKE is half the other?

**Report:** one table and two sentences.

In [ ]:
# Your code here


# Part C: open questions and AI critique

## C1 [Proposed]: is the leu calmer than it used to be?

**Question.** Has the volatility of EUR/RON changed between 2005–2012, 2013–2019 and 2020–2026, and can one GARCH model describe all three periods? Model: B1.

1. For each period compute the annualised sample volatility and the share of days with $|r_t| > 0.5\%$.
2. Estimate GARCH(1,1)-t on each period and report $\alpha + \beta$, $\alpha$ and $\nu$.
3. Find the largest daily move of each period and its date.
4. Interpretation: is the change in volatility a property of the market, or of the exchange-rate policy?

**Report:** a table and a plan for a project.

In [ ]:
# Your code here


## C2 [Proposed]: audit an AI answer

**Context.** An AI assistant interpreted a GARCH(1,1)-t fit of the BET (daily, since 2000). It answered:

- (a) the half-life of a volatility shock is ln 0.5 / ln beta;
- (b) the long-run variance is omega / (1 − alpha);
- (c) an AR(1) by OLS gives a t statistic of about 9, so BET returns are strongly predictable;
- (d) the Ljung–Box test of the squared standardised residuals does not reject, so the GARCH-t model is correct;
- (e) with Student-t innovations the 1% quantile of z is closer to 0 than −2.326, so the VaR 1% is smaller;
- (f) alpha + beta < 1, so the model is covariance stationary and the volatility reverts to its long-run level.

1. For each statement, say whether it is correct; if not, give the correct statement and the correct number.

**Report:** six verdicts with one line of justification each.

In [ ]:
# Your code here
